In [1]:
from fastapi import UploadFile
import polars as pl
import io
import re
from typing import Optional, Sequence
from dataclasses import dataclass, field
import chess
from chess import pgn

In [2]:
def create_uploadfile_from_path(file_path: str, filename: str) -> UploadFile:

    with open(file_path, "rb") as f:
        file_content = f.read()
        file_stream = io.BytesIO(file_content)

    return UploadFile(
            filename=filename,
            file=file_stream)

In [3]:
class ConvertToDf():

    @staticmethod
    def read_file(file: UploadFile) -> pl.DataFrame:
        """Read PGN file and convert to Polars DataFrame."""
        
        # Read file content
        content = file.file.read().decode('utf-8')
        
        # Parse PGN text
        games = ConvertToDf._parse_pgn(content)
        
        df = pl.DataFrame(games)

        df = ConvertToDf._apply_schema(df)

        df = ConvertToDf._add_move_count(df)
        
        return df
    
    @staticmethod
    def _parse_pgn(pgn_text: str) -> list[dict[str, str]]:
        """Parse PGN text into game dictionaries."""
        games: list[dict[str, str]] = []
        game_blocks = re.split(r'\n\n(?=\[Event)', pgn_text.strip())
        
        for block in game_blocks:
            if not block.strip():
                continue
            
            game_data = ConvertToDf._parse_game_block(block)
            if game_data:
                games.append(game_data)
        
        return games
    
    @staticmethod
    def _parse_game_block(block: str) -> Optional[dict[str, str]]:
        """Parse a single game block."""
        game_data: dict[str, str] = {}
        
        # Extract headers
        headers = re.findall(r'\[(\w+)\s+"([^"]*)"\]', block)
        for key, value in headers:
            game_data[key] = value
        
        # Extract moves
        moves_match = re.search(
            r'\]\n\n(.+?)(?:\s+(?:1-0|0-1|1/2-1/2|\*))?$',
            block,
            re.DOTALL
        )
        
        if moves_match:
            moves_text = moves_match.group(1).strip()
            # Remove comments in curly braces
            moves_clean = re.sub(r'\{[^}]*\}', '', moves_text)
            # Normalize whitespace
            moves_clean = re.sub(r'\s+', ' ', moves_clean).strip()
            game_data['Moves'] = moves_clean
        else:
            game_data['Moves'] = ''
        
        return game_data if game_data else None
    
    @staticmethod
    def _apply_schema(df: pl.DataFrame) -> pl.DataFrame:
        df = df.with_columns(pl.col('WhiteElo').cast(pl.Int16),
                             pl.col('BlackElo').cast(pl.Int16))
        return df
    
    @staticmethod
    def _count_moves(moves: str) -> int:
        """Count the number of full moves in a game."""
        game = pgn.read_game(io.StringIO(moves))
        if game is None:
           return 0
        return sum(1 for _ in game.mainline_moves()) // 2
    
    @staticmethod
    def _add_move_count(df: pl.DataFrame) -> pl.DataFrame:
        """Add move count column to dataframe."""
        return df.with_columns(
            pl.col('Moves')
            .map_elements(ConvertToDf._count_moves, return_dtype=pl.Int16)
            .alias('NumMoves')
        )
        


In [4]:
@dataclass
class FilterConfig:
    terminations: Sequence[str] = field(
        default_factory=lambda: ("Normal", "Time forfeit")
    )
    events: Sequence[str] = field(
        default_factory=lambda: (
            "Rated Blitz game",
            "Rated Bullet game",
            "Rated Rapid game",
        )
    )
    elo_range: tuple[int, int] = field(default_factory=lambda: (1000, 2000))
    min_moves: int = 15

filter_config = FilterConfig()


In [5]:
class GameFilter():

    @staticmethod
    def apply_filters(df: pl.DataFrame, config:FilterConfig) -> pl.DataFrame:
        df = GameFilter._filter_by_items_in_list(df, config.terminations,'Termination')
        df = GameFilter._filter_by_items_in_list(df, config.events, 'Event')
        df = GameFilter._filter_by_elo(df, config.elo_range)
        df = GameFilter._filter_by_move_count(df, config.min_moves)
        return df

    @staticmethod
    def _filter_by_elo(df: pl.DataFrame,
                    range: tuple[int, int]) -> pl.DataFrame:

        return df.filter(pl.col('WhiteElo').is_between(range[0], range[1]))

    @staticmethod
    def _filter_by_items_in_list(df: pl.DataFrame, 
                            list:Sequence[str], target_col: str) -> pl.DataFrame:
        pattern = '|'.join(list)
        return df.filter(pl.col(target_col).str.contains(pattern))

    @staticmethod
    def _filter_by_move_count(df: pl.DataFrame,
                           min_moves: int) -> pl.DataFrame:
        return df.filter(pl.col('NumMoves')>=min_moves)

In [6]:
INPUT = '../../../data/sample_data.pgn'
file = create_uploadfile_from_path(INPUT, 'sample_data.pgn')

In [7]:
df = ConvertToDf.read_file(file)
df_filtered = GameFilter.apply_filters(df, filter_config)

In [8]:
df_filtered.shape

(485, 20)

In [9]:
@dataclass
class MappingConfig:
    result_map: dict[str, int] = field(
        default_factory=lambda: {
            "1-0": 1,
            "0-1": -1,
            "1/2-1/2": 0,
        }
    )
    event_map: dict[str, str] = field(
        default_factory=lambda: {
            "Rated Blitz game": "Blitz",
            "Rated Bullet game": "Bullet",
            "Rated Rapid game": "Rapid",
        }
    )

mapping_config = MappingConfig()


In [10]:
class GameMapping():

    @staticmethod
    def apply_mappings(df: pl.DataFrame, config:MappingConfig) -> pl.DataFrame:
        df = GameMapping._map_result(df, config.result_map)
        df = GameMapping._map_event(df, config.event_map)
        return df

    @staticmethod
    def _map_result(df: pl.DataFrame,
                    mapping: dict[str, int]) -> pl.DataFrame:

        return df.with_columns(
            pl.col('Result').map_elements(lambda x: mapping.get(x, None), return_dtype=pl.Int8)
        )
    
    @staticmethod
    def _map_event(df: pl.DataFrame,
                   mapping: dict[str, str]) -> pl.DataFrame:

        return df.with_columns(
            pl.col('Event').map_elements(lambda x: mapping.get(x, None), return_dtype=pl.Utf8)
        )

In [11]:
df_mapped = GameMapping.apply_mappings(df_filtered, mapping_config)

In [12]:
df_mapped['Result'].value_counts()

Result,count
i8,u32
0,18
-1,220
1,247


In [13]:
@dataclass
class FeatureEngineerConfig:
    """Configuration for feature engineering."""
    move_number: int = 15  # Which move to analyze (full move number)
    piece_values: dict[int, int] = field(
        default_factory=lambda: {
            chess.PAWN: 1,
            chess.KNIGHT: 3,
            chess.BISHOP: 3,
            chess.ROOK: 5,
            chess.QUEEN: 9,
            chess.KING: 0
        }
    )
    center_squares: list[int] = field(
        default_factory=lambda: [chess.D4, chess.E4, chess.D5, chess.E5]
    )
    extended_center_squares: list[int] = field(
        default_factory=lambda: [
            chess.C3, chess.D3, chess.E3, chess.F3,
            chess.C4, chess.D4, chess.E4, chess.F4,
            chess.C5, chess.D5, chess.E5, chess.F5,
            chess.C6, chess.D6, chess.E6, chess.F6
        ]
    )



In [14]:
class FeatureEngineer():

    @staticmethod
    def add_features(df: pl.DataFrame, config: FeatureEngineerConfig) -> pl.DataFrame:
        """Add all chess features to the dataframe."""
        # Extract features for each game
        features_list: list[dict[str, float]] = []
        
        for moves in df['Moves']:
            board = FeatureEngineer._get_board_at_move(moves, config.move_number)

            features: dict[str, float] = {}
            features.update(FeatureEngineer._calculate_material(board, config))
            features.update(FeatureEngineer._calculate_attacked_pieces(board, config))
            features.update(FeatureEngineer._calculate_center_control(board, config))
            features.update(FeatureEngineer._calculate_castling_rights(board))
            features.update(FeatureEngineer._calculate_mobility(board, config))
            # Add weighted mobility for white and black as separate keys
            features['white_weighted_mobility'] = FeatureEngineer._calculate_weighted_mobility(board, chess.WHITE, config) if board is not None else 0.0
            features['black_weighted_mobility'] = FeatureEngineer._calculate_weighted_mobility(board, chess.BLACK, config) if board is not None else 0.0
            
            features_list.append(features)
        
        # Create DataFrame from features
        features_df = pl.DataFrame(features_list)
        
        # Concatenate with original dataframe
        return pl.concat([df, features_df], how="horizontal")
    
    @staticmethod
    def _get_board_at_move(moves: str, move_number: int) -> Optional[chess.Board]:
        """Get board state at a specific move number."""
        try:
            game = pgn.read_game(io.StringIO(moves))
            if game is None:
                return None
            
            board = game.board()
            move_count = 0
            
            for move in game.mainline_moves():
                board.push(move)
                move_count += 1
                # move_number * 2 because we count both white and black moves
                if move_count == move_number * 2:
                    return board
            
            # If game is shorter than requested move, return last position
            return board if move_count > 0 else None
        except:
            return None
    
    @staticmethod
    def _calculate_material(board: Optional[chess.Board], config: FeatureEngineerConfig) -> dict[str, int]:
        """Calculate material balance on the board."""
        white_material = 0
        black_material = 0

        if board is None:
            return {
                'white_material': 0,
                'black_material': 0,
                'material_diff': 0
            }
        else:
            for square in chess.SQUARES:
                piece = board.piece_at(square)
                if piece:
                    value = config.piece_values[piece.piece_type]
                    if piece.color == chess.WHITE:
                        white_material += value
                    else:
                        black_material += value
        
        return {
            'white_material': white_material,
            'black_material': black_material,
            'material_diff': white_material - black_material
        }
    
    @staticmethod
    def _calculate_attacked_pieces(board: Optional[chess.Board], config: FeatureEngineerConfig) -> dict[str, int]:
        """Calculate number and value of pieces under attack."""
        if board is None:
            return {
                'white_pieces_attacked': 0,
                'white_attacked_value': 0,
                'black_pieces_attacked': 0,
                'black_attacked_value': 0,
                'attacked_diff': 0
            }
        
        white_attacked_count = 0
        white_attacked_value = 0
        black_attacked_count = 0
        black_attacked_value = 0
        
        for square in chess.SQUARES:
            piece = board.piece_at(square)
            if piece:
                is_attacked = board.is_attacked_by(not piece.color, square)
                
                if is_attacked:
                    piece_value = config.piece_values[piece.piece_type]
                    
                    if piece.color == chess.WHITE:
                        white_attacked_count += 1
                        white_attacked_value += piece_value
                    else:
                        black_attacked_count += 1
                        black_attacked_value += piece_value
        
        return {
            'white_pieces_attacked': white_attacked_count,
            'white_attacked_value': white_attacked_value,
            'black_pieces_attacked': black_attacked_count,
            'black_attacked_value': black_attacked_value,
            'attacked_diff': black_attacked_value - white_attacked_value
        }
    
    @staticmethod
    def _calculate_center_control(board: Optional[chess.Board], config: FeatureEngineerConfig) -> dict[str, int]:
        """Calculate center control metrics."""
        if board is None:
            return {
                'white_center_pieces': 0,
                'black_center_pieces': 0,
                'white_center_control': 0,
                'black_center_control': 0,
                'center_control_diff': 0,
                'white_extended_control': 0,
                'black_extended_control': 0,
                'extended_control_diff': 0
            }
        
        # Center occupation
        white_center_pieces = 0
        black_center_pieces = 0
        
        for square in config.center_squares:
            piece = board.piece_at(square)
            if piece:
                if piece.color == chess.WHITE:
                    white_center_pieces += 1
                else:
                    black_center_pieces += 1
        
        # Center attack/control
        white_center_attacks = sum(
            1 for sq in config.center_squares 
            if board.is_attacked_by(chess.WHITE, sq)
        )
        black_center_attacks = sum(
            1 for sq in config.center_squares 
            if board.is_attacked_by(chess.BLACK, sq)
        )
        
        # Extended center control
        white_extended_attacks = sum(
            1 for sq in config.extended_center_squares 
            if board.is_attacked_by(chess.WHITE, sq)
        )
        black_extended_attacks = sum(
            1 for sq in config.extended_center_squares 
            if board.is_attacked_by(chess.BLACK, sq)
        )
        
        return {
            'white_center_pieces': white_center_pieces,
            'black_center_pieces': black_center_pieces,
            'white_center_control': white_center_attacks,
            'black_center_control': black_center_attacks,
            'center_control_diff': white_center_attacks - black_center_attacks,
            'white_extended_control': white_extended_attacks,
            'black_extended_control': black_extended_attacks,
            'extended_control_diff': white_extended_attacks - black_extended_attacks
        }
    
    @staticmethod
    def _calculate_castling_rights(board: Optional[chess.Board]) -> dict[str, int]:
        """Calculate castling rights status."""
        if board is None:
            return {
                'white_can_castle_kingside': 0,
                'white_can_castle_queenside': 0,
                'black_can_castle_kingside': 0,
                'black_can_castle_queenside': 0,
                'white_has_castled': 0,
                'black_has_castled': 0,
                'white_castling_rights_count': 0,
                'black_castling_rights_count': 0
            }
        
        # Check if kings have moved (indicating castling has occurred)
        white_king_sq = board.king(chess.WHITE)
        black_king_sq = board.king(chess.BLACK)
        
        white_has_castled = (
            not board.has_castling_rights(chess.WHITE) and
            white_king_sq is not None and
            white_king_sq in [chess.G1, chess.C1]
        )
        
        black_has_castled = (
            not board.has_castling_rights(chess.BLACK) and
            black_king_sq is not None and
            black_king_sq in [chess.G8, chess.C8]
        )
        
        return {
            'white_can_castle_kingside': int(board.has_kingside_castling_rights(chess.WHITE)),
            'white_can_castle_queenside': int(board.has_queenside_castling_rights(chess.WHITE)),
            'black_can_castle_kingside': int(board.has_kingside_castling_rights(chess.BLACK)),
            'black_can_castle_queenside': int(board.has_queenside_castling_rights(chess.BLACK)),
            'white_has_castled': int(white_has_castled),
            'black_has_castled': int(black_has_castled),
            'white_castling_rights_count': (
                board.has_kingside_castling_rights(chess.WHITE) +
                board.has_queenside_castling_rights(chess.WHITE)
            ),
            'black_castling_rights_count': (
                board.has_kingside_castling_rights(chess.BLACK) +
                board.has_queenside_castling_rights(chess.BLACK)
            )
        }
    
    @staticmethod
    def _calculate_mobility(board: Optional[chess.Board], config: FeatureEngineerConfig) -> dict[str, float]:
        """Calculate mobility (degrees of freedom) metrics."""
        if board is None:
            return {
                'white_mobility': 0,
                'black_mobility': 0,
                'mobility_diff': 0,
                'white_weighted_mobility': 0.0,
                'black_weighted_mobility': 0.0,
                'weighted_mobility_diff': 0.0
            }
        
        # Count legal moves for each side
        current_turn = board.turn
        
        # White mobility
        if current_turn == chess.WHITE:
            white_legal_moves = board.legal_moves.count()
            board.turn = chess.BLACK
            black_legal_moves = board.legal_moves.count()
            board.turn = chess.WHITE
        else:
            black_legal_moves = board.legal_moves.count()
            board.turn = chess.WHITE
            white_legal_moves = board.legal_moves.count()
            board.turn = chess.BLACK
        
        # Calculate weighted mobility
        white_weighted = FeatureEngineer._calculate_weighted_mobility(board, chess.WHITE, config)
        black_weighted = FeatureEngineer._calculate_weighted_mobility(board, chess.BLACK, config)
        
        return {
            'white_mobility': white_legal_moves,
            'black_mobility': black_legal_moves,
            'mobility_diff': white_legal_moves - black_legal_moves,
            'white_weighted_mobility': white_weighted,
            'black_weighted_mobility': black_weighted,
            'weighted_mobility_diff': white_weighted - black_weighted
        }
    
    @staticmethod
    def _calculate_weighted_mobility(board: chess.Board, color: bool, config: FeatureEngineerConfig) -> float:
        """Calculate weighted mobility (sum of legal moves × piece value)."""
        weighted_sum = 0.0
        
        for square in chess.SQUARES:
            piece = board.piece_at(square)
            if piece and piece.color == color:
                piece_value = config.piece_values[piece.piece_type]
                moves_from_square = sum(
                    1 for move in board.legal_moves 
                    if move.from_square == square
                )
                weighted_sum += moves_from_square * piece_value
        
        return weighted_sum


In [15]:
df_features = FeatureEngineer.add_features(df_mapped, FeatureEngineerConfig())

In [16]:
df.columns

['Event',
 'Site',
 'Date',
 'Round',
 'White',
 'Black',
 'Result',
 'UTCDate',
 'UTCTime',
 'WhiteElo',
 'BlackElo',
 'WhiteRatingDiff',
 'BlackRatingDiff',
 'ECO',
 'Opening',
 'TimeControl',
 'Termination',
 'Moves',
 'BlackTitle',
 'NumMoves']

In [17]:
df_features.sample()

Event,Site,Date,Round,White,Black,Result,UTCDate,UTCTime,WhiteElo,BlackElo,WhiteRatingDiff,BlackRatingDiff,ECO,Opening,TimeControl,Termination,Moves,BlackTitle,NumMoves,white_material,black_material,material_diff,white_pieces_attacked,white_attacked_value,black_pieces_attacked,black_attacked_value,attacked_diff,white_center_pieces,black_center_pieces,white_center_control,black_center_control,center_control_diff,white_extended_control,black_extended_control,extended_control_diff,white_can_castle_kingside,white_can_castle_queenside,black_can_castle_kingside,black_can_castle_queenside,white_has_castled,black_has_castled,white_castling_rights_count,black_castling_rights_count,white_mobility,black_mobility,mobility_diff,white_weighted_mobility,black_weighted_mobility,weighted_mobility_diff
str,str,str,str,str,str,i8,str,str,i16,i16,str,str,str,str,str,str,str,str,i16,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,f64,f64
"""Bullet""","""https://lichess.org/v0MT9GEo""","""2025.07.01""","""-""","""percy1234""","""oscargame""",-1,"""2025.07.01""","""00:00:23""",1993,2139,"""-4""","""+4""","""D00""","""Queen's Pawn Game""","""120+1""","""Time forfeit""","""1. d4 1... d5 2. e3 2... e6 3.…",null,47,36,35,1,1,1,4,4,3,2,1,3,2,1,12,11,1,0,0,0,0,1,0,0,0,42,24,18,174.0,0.0,174.0


In [18]:
@dataclass
class SelectionConfig:
    features_to_drop: Sequence[str] = field(default_factory=lambda: [
        'Site',
        'Date',
        'Round',
        'White',
        'Black',
        'UTCDate',
        'UTCTime',
        'BlackTitle',
        'NumMoves',
        'Moves',
        'WhiteRatingDiff',
        'BlackRatingDiff',
        
    ])

In [19]:
class FeatureSelector():
    
    @staticmethod
    def select_features(df: pl.DataFrame, config:SelectionConfig) -> pl.DataFrame:
        return df.drop(config.features_to_drop)

In [20]:
df_selected = FeatureSelector.select_features(df_features, SelectionConfig())
df_selected.sample(5)

Event,Result,WhiteElo,BlackElo,ECO,Opening,TimeControl,Termination,white_material,black_material,material_diff,white_pieces_attacked,white_attacked_value,black_pieces_attacked,black_attacked_value,attacked_diff,white_center_pieces,black_center_pieces,white_center_control,black_center_control,center_control_diff,white_extended_control,black_extended_control,extended_control_diff,white_can_castle_kingside,white_can_castle_queenside,black_can_castle_kingside,black_can_castle_queenside,white_has_castled,black_has_castled,white_castling_rights_count,black_castling_rights_count,white_mobility,black_mobility,mobility_diff,white_weighted_mobility,black_weighted_mobility,weighted_mobility_diff
str,i8,i16,i16,str,str,str,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,f64,f64
"""Bullet""",-1,1002,1051,"""B00""","""Pirc Defense""","""60+0""","""Time forfeit""",29,29,0,3,5,1,1,-4,0,2,3,2,1,9,12,-3,1,1,1,0,0,0,2,1,43,40,3,199.0,0.0,199.0
"""Rapid""",1,1152,1160,"""B90""","""Sicilian Defense: Najdorf Vari…","""600+0""","""Normal""",31,30,1,4,8,3,7,-1,1,2,2,3,-1,9,10,-1,1,1,0,0,0,1,2,0,37,36,1,134.0,0.0,134.0
"""Blitz""",0,1262,1204,"""C00""","""French Defense""","""300+0""","""Normal""",35,35,0,3,9,1,1,-8,0,1,4,3,1,9,12,-3,0,0,0,0,1,1,0,0,35,40,-5,153.0,0.0,153.0
"""Blitz""",1,1578,1588,"""C00""","""French Defense""","""300+0""","""Normal""",34,36,-2,3,3,5,17,14,1,1,3,3,0,12,12,0,1,1,0,0,0,1,2,0,54,42,12,220.0,0.0,220.0
"""Blitz""",1,1285,1304,"""D05""","""Queen's Pawn Game: Colle Syste…","""180+0""","""Time forfeit""",22,23,-1,1,3,1,1,-2,1,1,4,1,3,8,9,-1,1,1,0,0,0,0,2,0,24,31,-7,54.0,0.0,54.0


In [21]:
df_selected.schema

Schema([('Event', String),
        ('Result', Int8),
        ('WhiteElo', Int16),
        ('BlackElo', Int16),
        ('ECO', String),
        ('Opening', String),
        ('TimeControl', String),
        ('Termination', String),
        ('white_material', Int64),
        ('black_material', Int64),
        ('material_diff', Int64),
        ('white_pieces_attacked', Int64),
        ('white_attacked_value', Int64),
        ('black_pieces_attacked', Int64),
        ('black_attacked_value', Int64),
        ('attacked_diff', Int64),
        ('white_center_pieces', Int64),
        ('black_center_pieces', Int64),
        ('white_center_control', Int64),
        ('black_center_control', Int64),
        ('center_control_diff', Int64),
        ('white_extended_control', Int64),
        ('black_extended_control', Int64),
        ('extended_control_diff', Int64),
        ('white_can_castle_kingside', Int64),
        ('white_can_castle_queenside', Int64),
        ('black_can_castle_kingside', Int6

In [22]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import matthews_corrcoef, confusion_matrix

from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

In [23]:
X = df_selected.drop('Result')
y = df_selected['Result']

In [24]:
ohe_cols = ['Event','TimeControl', 'Termination']

ordinal_cols = ['ECO','Opening']

numeric_cols =[ 'WhiteElo', 'BlackElo', 'white_material','black_material',
        'material_diff', 'white_pieces_attacked',
        'white_attacked_value', 'black_pieces_attacked',
        'black_attacked_value','attacked_diff',
        'white_center_pieces', 'black_center_pieces',
        'white_center_control', 'black_center_control',
        'center_control_diff', 'white_extended_control',
        'black_extended_control', 'extended_control_diff',
        'white_can_castle_kingside', 'white_can_castle_queenside',
        'black_can_castle_kingside', 'black_can_castle_queenside',
        'white_has_castled', 'black_has_castled',
        'white_castling_rights_count', 'black_castling_rights_count',
        'white_mobility', 'black_mobility',
        'mobility_diff', 'white_weighted_mobility',
        'black_weighted_mobility', 'weighted_mobility_diff']

In [31]:
preprecessing_pipeline = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numeric_cols),
    ('ord', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), ordinal_cols),
    ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ohe_cols)
])

logistic_model = Pipeline(steps=[
    ('preprocessor', preprecessing_pipeline),
    ('classifier', LogisticRegression(max_iter=1000))
])

random_forest_model = Pipeline(steps=[
    ('preprocessor', preprecessing_pipeline),
    ('classifier', RandomForestClassifier(n_estimators=50, random_state=42))
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


In [27]:
logistic_model.fit(X_train, y_train)

y_pred_log = logistic_model.predict(X_test)

metrics_log = {
    'matthews_corrcoef': matthews_corrcoef(y_test, y_pred_log),
    'confusion_matrix': confusion_matrix(y_test, y_pred_log) }

metrics_log

/home/ge/MCD/Tesis/Tesis/fastapi-production-template/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


{'matthews_corrcoef': 0.1620302651541228,
 'confusion_matrix': array([[18,  0, 26],
        [ 1,  0,  3],
        [12,  0, 37]])}

In [32]:
random_forest_model.fit(X_train, y_train)

y_pred_rf = random_forest_model.predict(X_test)

metrics_rf = {
    'matthews_corrcoef': matthews_corrcoef(y_test, y_pred_rf),
    'confusion_matrix': confusion_matrix(y_test, y_pred_rf) }

metrics_rf

{'matthews_corrcoef': 0.11948966967363624,
 'confusion_matrix': array([[17,  0, 27],
        [ 2,  0,  2],
        [13,  0, 36]])}